# Eksplorasi Magnetoimpedansi — interaktif (`ipywidgets`)

Notebook pendamping `simulator_magnetoimpedansi.py` (port Python dari `simulator-magnetoimpedansi-teoretis.html`).
Geser-geser slider di bawah untuk melihat kurva `MIR(H)`, puncak, sensitivitas, dan LOD secara langsung.
Bagian 5–7: pita ketidakpastian ±1σ, ekspor/impor CSV pengukuran (`measure_csv`/`parse_exp_csv`), dan contoh fit `fit_sensor`/`fit_cd`.

**Cara pakai:** pilih kernel **Python (nr)**, lalu *Run All*. Butuh `scipy`, `ipywidgets`, `matplotlib` — semua sudah ada di env `nr`.

In [ ]:
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
import scipy
import scipy.constants as const
import scipy.special as sps
import ipywidgets as W

import simulator_magnetoimpedansi as sim

print("numpy", np.__version__, "| scipy", scipy.__version__, "| ipywidgets", W.__version__)

## 1. Cek silang `scipy` vs konstanta/tabel hardcode JS

Memastikan tabel kuadratur `GH_X/W` dan `Phi` di portingan cocok dengan `scipy` (sumber independen).

In [ ]:
# mu0 JS memakai 4*pi*1e-7 eksak; CODATA (SI 2019) sedikit berbeda — bandingkan relatif
dmu = abs(sim.MU0 - const.mu_0) / const.mu_0
print(f"MU0 sim={sim.MU0:.6e}  scipy={const.mu_0:.6e}  beda-rel={dmu:.3e}")
assert dmu < 1e-9, dmu
dk = abs(sim.KB - const.k) / const.k
print(f"KB  sim={sim.KB:.6e}  scipy={const.k:.6e}  beda-rel={dk:.3e}")
assert dk < 1e-15, dk

# GH fisikawan ternormalisasi: X = sqrt(2)*x_fis, W = w_fis/sqrt(pi)
x_fis, w_fis = sps.roots_hermite(7)
order = np.argsort(x_fis)
X_sp = (np.sqrt(2) * x_fis[order]).tolist()
W_sp = ((w_fis / np.sqrt(np.pi))[order]).tolist()
dX = max(abs(a - b) for a, b in zip(X_sp, sim.GH_X))
dW = max(abs(a - b) for a, b in zip(W_sp, sim.GH_W))
print(f"maks |dX| vs tabel JS = {dX:.3e}")
print(f"maks |dW| vs tabel JS = {dW:.3e}")
assert dX < 1e-6 and dW < 1e-6, (dX, dW)
print(f"sum(W) = {sum(sim.GH_W):.6f} (harus 1.0)")
assert abs(sum(sim.GH_W) - 1.0) < 1e-6

# Phi(math.erf) vs Phi(scipy ndtr) di beberapa titik
for x in (-2.0, -0.5, 0.0, 0.3, 2.5):
    assert abs(sim.phi_norm(x) - sps.ndtr(x)) < 1e-15, x
print("Phi OK (selisih < 1e-15 di semua titik uji)")

## 2. Penjelajah kurva `MIR(H)` — geser slidernya

In [ ]:
def _mir_explorer(mat="fe3o4", f_log=1.0, Hk=3.0, mass=0.10, Hc=0.2, axis="long", step=0.5):
    P = sim.default_P()
    P.update(dict(Hk=Hk, Hc=Hc, axis=axis, f=10 ** f_log))
    m = sim.default_mat(mat)
    masses = [0.0, mass] if mass > 0 else [0.0]
    fig, ax = plt.subplots(figsize=(8, 4.6))
    curves = []
    for mm in masses:
        c = sim.model_curve(P, m, mm, "up", step)
        curves.append(c)
        info = sim.peak_info(c)
        ax.plot([p["x"] for p in c], [p["y"] for p in c], label=f"{mm:g} g (puncak {info['peak']:.2f}%)")
    ax.set_xlabel("H (mT)")
    ax.set_ylabel("MIR (%)")
    ax.set_title(f"{m['name']} @ {P['f']:.1f} MHz, Hk={Hk} mT, sumbu={axis}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
    if len(curves) == 2:
        d = sim.peak_info(curves[1])["peak"] - sim.peak_info(curves[0])["peak"]
        print(f"delta puncak vs acuan m=0: {d:+.3f} pp")


W.interact(
    _mir_explorer,
    mat=W.Dropdown(options=sorted(sim.MATS), value="fe3o4", description="Bahan"),
    f_log=W.FloatSlider(min=0, max=3, step=0.05, value=1.0, description="log f [MHz]"),
    Hk=W.FloatSlider(min=0.5, max=10, step=0.1, value=3.0, description="Hk [mT]"),
    mass=W.FloatSlider(min=0, max=0.5, step=0.05, value=0.10, description="massa [g]"),
    Hc=W.FloatSlider(min=0, max=1, step=0.05, value=0.2, description="Hc [mT]"),
    axis=W.Dropdown(options=["long", "trans"], value="long", description="sumbu"),
    step=W.FloatSlider(min=0.25, max=2.0, step=0.25, value=0.5, description="langkah"),
);

## 3. Sensitivitas & batas deteksi (LOD)

In [ ]:
def _lod_explorer(mat="fe3o4", f_log=1.0, Hk=3.0, m_max=0.30):
    P = sim.default_P()
    P.update(dict(Hk=Hk, f=10 ** f_log))
    m = sim.default_mat(mat)
    ms = np.linspace(0, m_max, 13).tolist()
    peaks = [sim.peak_at(P, m, x) for x in ms]
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(ms, peaks, "o-")
    ax.set_xlabel("massa (g)")
    ax.set_ylabel("puncak MIR (%)")
    ax.grid(True, alpha=0.3)
    plt.show()
    sl = sim.sensitivity_lod(P, m)
    print(f"S = {sl['S']:.2f} %/g   LOD = {sl['lod'] * 1000:.2f} mg   (dm={sl['dm']:.5f} g)")


W.interact(
    _lod_explorer,
    mat=W.Dropdown(options=sorted(sim.MATS), value="fe3o4", description="Bahan"),
    f_log=W.FloatSlider(min=0, max=3, step=0.05, value=1.0, description="log f [MHz]"),
    Hk=W.FloatSlider(min=0.5, max=10, step=0.1, value=3.0, description="Hk [mT]"),
    m_max=W.FloatSlider(min=0.1, max=1.0, step=0.05, value=0.30, description="m maks [g]"),
);

## 4. Uji mandiri (tombol)

In [ ]:
btn = W.Button(description="Jalankan uji mandiri", button_style="primary")
out = W.Output()


def _run(_):
    with out:
        out.clear_output()
        results = sim.self_test(sim.default_P())
        for name, ok, detail in results:
            print(f"[{'OK' if ok else 'GAGAL'}] {name} -- {detail}")
        print(f"\n{sum(1 for _, ok, _ in results if ok)}/{len(results)} lolos")


btn.on_click(_run)
display(btn, out)

## 5. Pita ketidakpastian ±1σ & komponen impedansi

`model_band(c, P)` memakai `slope_at` + `sigma_mir` (noise baca `sigHr` & `sigZ`). Mode MIR menampilkan pita ±1σ; mode Komponen Z menampilkan Re/Im/|Z|.

In [ ]:
def _band_explorer(mat="fe3o4", f_log=1.0, Hk=3.0, mass=0.10, mode="MIR + pita"):
    P = sim.default_P()
    P.update(dict(Hk=Hk, f=10 ** f_log))
    m = sim.default_mat(mat)
    c = sim.model_curve(P, m, mass, "up", 0.25)
    band = sim.model_band(c, P, "read")
    x = [p["x"] for p in c]
    fig, ax = plt.subplots(figsize=(8, 4.6))
    if mode.startswith("MIR"):
        ax.plot(x, [p["y"] for p in c], label="MIR model")
        ax.fill_between(x, [p["y"] for p in band["lo"]], [p["y"] for p in band["hi"]],
                        alpha=0.25, label="pita ±1σ")
        ax.set_ylabel("MIR (%)")
    else:
        ax.plot(x, [p["re"] for p in c], label="Re Z")
        ax.plot(x, [p["im"] for p in c], label="Im Z")
        ax.plot(x, [p["abs"] for p in c], label="|Z|")
        ax.set_ylabel("Impedansi (ohm)")
    ax.set_xlabel("H (mT)")
    ax.set_title(f"{m['name']} @ {P['f']:.1f} MHz, massa {mass:g} g")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


W.interact(
    _band_explorer,
    mat=W.Dropdown(options=sorted(sim.MATS), value="fe3o4", description="Bahan"),
    f_log=W.FloatSlider(min=0, max=3, step=0.05, value=1.0, description="log f [MHz]"),
    Hk=W.FloatSlider(min=0.5, max=10, step=0.1, value=3.0, description="Hk [mT]"),
    mass=W.FloatSlider(min=0, max=0.5, step=0.05, value=0.10, description="massa [g]"),
    mode=W.Dropdown(options=["MIR + pita", "Komponen Z"], value="MIR + pita", description="mode"),
);

## 6. CSV pengukuran: ekspor, impor, dan clamp

`measure_csv` meniru unduhan CSV halaman HTML (10 kolom). `parse_exp_csv` mengembalikan `(rows, bad)` dan tahan desimal koma, pemisah `,`/`;`/tab, dan kutip. `clamp_P`/`clamp_mat` meniru pembatasan slider `getP`/`getMat`.

In [ ]:
P = sim.default_P()
m = sim.default_mat("fe3o4")

csv_text = sim.measure_csv(P, m, [0.0, 0.1], seed=20261012)
print("\n".join(csv_text.splitlines()[:3]) + "   ...")
rows, bad = sim.parse_exp_csv(csv_text)
print(f"impor balik: {len(rows)} baris, {bad} rusak, label contoh = {rows[1]['label']!r}")

cp = sim.clamp_P(dict(P, pf=9, edgeC=5, T=-3, Hfine=999))
cm = sim.clamp_mat(dict(m, d=0.01, sigD=-1))
assert (cp["pf"], cp["edgeC"], cp["T"], cp["Hfine"]) == (0.64, 1, 1, P["Hmax"])
assert (cm["d"], cm["sigD"]) == (0.5, 0)
sp = sim.setpoints(P)
print(f"clamp OK; setpoints: {len(sp)} titik, 0 dan {P['Hmax']:g} mT ada: {0.0 in sp and P['Hmax'] in sp}")

## 7. Data sintetis & fit (`fit_sensor`, `fit_cd`, `nelder_mead`)

Label eksperimen dibuat dari model + noise RNG deterministik (Hk asli 4 mT, Cd asli 0.15), lalu direfit mulai dari titik awal 3 mT / 0.1 lalu diplot.

In [ ]:
P_fit = sim.default_P()
P_fit.update(dict(sigZ=0.05, sigHr=0.02))
m_fit = sim.default_mat("fe3o4")
mass = 0.10
P_true = dict(P_fit, Hk=4.0, Cd=0.15)
rnd = sim.Randn(7)
exp_pts = [dict(label="0.1 gram", H=h,
                mir=sim.model_mir(P_true, m_fit, mass, h) + 0.06 * rnd())
           for h in np.linspace(-6, 6, 41).tolist()]

r1 = sim.fit_sensor(P_fit, m_fit, "0.1 gram", exp_pts, mass=mass)
fit_P = dict(P_fit, Hk=r1["x"][0], sigHk=r1["x"][1])
r2 = sim.fit_cd(fit_P, m_fit, exp_pts, [dict(label="0.1 gram", mass=mass)])
print(f"Hk asli 4.000 -> {r1['x'][0]:.3f} mT | sigHk asli 0 -> {r1['x'][1]:.3f} (cost={r1['fun']:.4f})")
print(f"Cd asli 0.150 -> {r2['x'][0]:.4f} (cost={r2['fun']:.4f})")

c_fit = sim.model_curve(dict(fit_P, Cd=r2["x"][0]), m_fit, mass, "up", 0.25)
res = [d["mir"] - sim.model_mir(dict(fit_P, Cd=r2["x"][0]), m_fit, mass, d["H"]) for d in exp_pts]
print(f"maks |residu| vs model fit (mir): {max(map(abs, res)):.3f} pp")
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot([p["x"] for p in c_fit], [p["y"] for p in c_fit], label="model hasil fit (sapuan up)")
ax.plot([d["H"] for d in exp_pts], [d["mir"] for d in exp_pts], "o", ms=3.5, label="eksperimen sintetis")
ax.set_xlabel("H (mT)")
ax.set_ylabel("MIR (%)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()